# Cell 01 - Load Dense Index và corpus cho Naive RAG

## Mục tiêu cell này
Load lại FAISS index, dense metadata và embedding model để xây dựng phiên bản Naive RAG đầu tiên.

## Vì sao cần làm bước này?
Ở notebook 05, ta đã tạo Dense Retrieval index bằng FAISS.  
Trong notebook 06, ta sẽ dùng index đó để truy xuất context cho câu hỏi người dùng.

Naive RAG nghĩa là pipeline đơn giản nhất:

User question  
→ Dense Retrieval tìm top-k chunks  
→ Ghép context vào prompt  
→ LLM sinh câu trả lời có nguồn

Đây là baseline RAG đầu tiên trước khi nâng cấp lên Hybrid RAG và Reranker.

## Giải thích code
Code sẽ:
1. Xác định đúng thư mục project
2. Load `dense_metadata.csv`
3. Load `faiss_index.index`
4. Load embedding model `intfloat/multilingual-e5-base`
5. Kiểm tra số vector trong FAISS index có khớp số dòng metadata không

## Output mong đợi
Bạn cần thấy:
- metadata khoảng 91,448 dòng
- FAISS index có 91,448 vectors
- model embedding chạy trên GPU nếu CUDA khả dụng

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import faiss
import torch
from sentence_transformers import SentenceTransformer

cwd = Path.cwd().resolve()
PROJECT = cwd.parent if cwd.name == "notebooks" else cwd

FAISS_DIR = PROJECT / "indexes" / "faiss"
ARTIFACT_DIR = PROJECT / "artifacts"
PROMPT_DIR = ARTIFACT_DIR / "prompts"

PROMPT_DIR.mkdir(parents=True, exist_ok=True)

metadata_path = FAISS_DIR / "dense_metadata.csv"
faiss_index_path = FAISS_DIR / "faiss_index.index"

chunks_df = pd.read_csv(metadata_path, low_memory=False)

id_cols = ["corpus_id", "chunk_id", "parent_id", "source_type", "title", "source_path", "language"]

for col in id_cols:
    chunks_df[col] = chunks_df[col].fillna("").astype(str)

chunks_df["retrieval_text"] = chunks_df["retrieval_text"].fillna("").astype(str)
chunks_df["chunk_text"] = chunks_df["chunk_text"].fillna("").astype(str)

faiss_index = faiss.read_index(str(faiss_index_path))

device = "cuda" if torch.cuda.is_available() else "cpu"
EMBEDDING_MODEL_NAME = "intfloat/multilingual-e5-base"

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=device)

print("Project:", PROJECT)
print("Chunks metadata:", chunks_df.shape)
print("FAISS vectors:", faiss_index.ntotal)
print("Embedding model:", EMBEDDING_MODEL_NAME)
print("Device:", device)

assert len(chunks_df) == faiss_index.ntotal, "Metadata và FAISS index không khớp số lượng."

W0510 00:06:05.534000 24332 torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Project: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag
Chunks metadata: (91448, 11)
FAISS vectors: 91448
Embedding model: intfloat/multilingual-e5-base
Device: cuda


# Cell 02 - Tạo hàm Dense Retrieval cho Naive RAG

## Mục tiêu cell này
Tạo hàm truy xuất các chunks liên quan nhất từ FAISS Dense Index dựa trên câu hỏi người dùng.

## Vì sao cần làm bước này?
Naive RAG là phiên bản RAG đơn giản nhất:

User question  
→ Dense Retrieval  
→ lấy top-k context  
→ đưa context vào prompt  
→ LLM sinh câu trả lời

Ở bước này, ta chưa gọi LLM.  
Ta chỉ kiểm tra phần Retrieval: câu hỏi đưa vào có lấy được đúng context/evidence hay không.

## Giải thích code
Code sẽ:
1. Thêm prefix `query:` cho câu hỏi theo chuẩn E5
2. Encode câu hỏi thành vector embedding
3. Search trong FAISS index
4. Lấy top-k chunks liên quan nhất
5. Trả về metadata như `source_type`, `title`, `parent_id`, `score`, `chunk_text`

Hàm có tham số `source_filter` để có thể truy xuất:
- chỉ legal corpus
- chỉ company handbook
- hoặc cả hai nguồn

## Output mong đợi
Bạn cần thấy kết quả truy xuất thử cho một câu hỏi về handbook nội bộ công ty.

In [2]:
def dense_retrieve_chunks(query, top_k=5, source_filter=None, search_k=100):
    query_embedding = embedding_model.encode(
        ["query: " + str(query)],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    ).astype("float32")

    scores, indices = faiss_index.search(query_embedding, search_k)

    rows = []
    rank = 1

    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue

        row = chunks_df.iloc[idx]

        if source_filter is not None and row["source_type"] not in source_filter:
            continue

        rows.append({
            "rank": rank,
            "score": float(score),
            "corpus_id": row["corpus_id"],
            "chunk_id": row["chunk_id"],
            "parent_id": row["parent_id"],
            "source_type": row["source_type"],
            "title": row["title"],
            "language": row["language"],
            "chunk_text": row["chunk_text"]
        })

        rank += 1

        if len(rows) >= top_k:
            break

    return pd.DataFrame(rows)

query = "What is the company policy for managing work devices?"

retrieved_df = dense_retrieve_chunks(
    query=query,
    top_k=5,
    source_filter=["company_handbook"]
)

display(retrieved_df[["rank", "score", "source_type", "title", "chunk_text"]])

,rank,score,source_type,title,chunk_text
0,1,0.825828,company_handbook,Managing Work Devices,"With Linux, we run Omarchy, which already come..."
1,2,0.825389,company_handbook,Managing Work Devices,# Managing work devices\nEveryone receives a n...
2,3,0.819602,company_handbook,Managing Work Devices,## Mobile devices and Windows\nDevices running...
3,4,0.788421,company_handbook,Moonlighting,## Not OK\n1. You can’t work full time or part...
4,5,0.783055,company_handbook,Readme,Read on basecamp.com\n\n# 37signals Employee H...


# Cell 03 - Tạo hàm format retrieved context cho prompt RAG

## Mục tiêu cell này
Chuyển các chunks truy xuất được thành phần `context` rõ ràng để đưa vào prompt cho LLM.

## Vì sao cần làm bước này?
RAG không chỉ truy xuất tài liệu, mà còn phải đưa tài liệu đó vào prompt để LLM trả lời dựa trên nguồn.

Nếu context được format lộn xộn, LLM khó biết:
- nguồn nào là handbook nội bộ
- nguồn nào là pháp luật
- đoạn nào là evidence
- chunk nào được dùng để trả lời

Vì vậy ta cần format context theo cấu trúc rõ ràng.

## Giải thích code
Code sẽ tạo hàm `format_context()`:
1. Duyệt từng chunk được retrieval
2. Gắn số thứ tự nguồn `[Source 1]`, `[Source 2]`, ...
3. Hiển thị metadata:
   - source_type
   - title
   - parent_id
   - chunk_id
   - score
4. Thêm nội dung `chunk_text`

Kết quả format này sẽ được dùng trong prompt ở cell sau.

## Output mong đợi
Bạn cần thấy context được trình bày rõ ràng theo từng source.

In [3]:
def format_context(retrieved_df, max_chars_per_chunk=1200):
    context_blocks = []

    for _, row in retrieved_df.iterrows():
        chunk_text = str(row["chunk_text"])[:max_chars_per_chunk]

        block = f"""
[Source {int(row["rank"])}]
source_type: {row["source_type"]}
title: {row["title"]}
parent_id: {row["parent_id"]}
chunk_id: {row["chunk_id"]}
score: {row["score"]:.4f}

{chunk_text}
""".strip()

        context_blocks.append(block)

    return "\n\n---\n\n".join(context_blocks)

context_text = format_context(retrieved_df)

print(context_text[:3000])

[Source 1]
source_type: company_handbook
title: Managing Work Devices
parent_id: company_0004
chunk_id: company_0004_001
score: 0.8258

With Linux, we run Omarchy, which already comes with the standard configuration needed (full-disk encryption, firewall, etc). Here we lean on 1password to provide the employee onboarding/offboarding of access to credentials and our Tailscale VPN to control access to internal systems. We don't use a managed process like Kandji.

## Access to code and secrets
Knowing our devices are safe and secure allows us to entrust our work computers with access to sensitive systems like Queenbee, and our internal VPN and remote servers. This means installing the VPN, checking out 37signals code, and storing secrets must only be done on a work device, not a personal device.

Please do not keep any personal data on your 37signals-issued computer. You should maintain a separate, personally-owned machine if you need a home computer. The company reserves the right to, an

# Cell 04 - Tạo prompt template cho Naive RAG

## Mục tiêu cell này
Tạo prompt chuẩn cho phiên bản Naive RAG.

## Vì sao cần làm bước này?
RAG không chỉ truy xuất context, mà còn cần đưa context vào prompt để LLM trả lời đúng nguồn.

Nếu prompt không ràng buộc rõ, LLM có thể:
- trả lời lan man
- tự suy đoán ngoài tài liệu
- không trích nguồn
- không biết từ chối khi context không đủ thông tin

Vì vậy prompt cần yêu cầu LLM:
1. Chỉ trả lời dựa trên retrieved context
2. Không bịa thông tin ngoài context
3. Trích dẫn source đã dùng
4. Nếu context không đủ thì phải nói chưa đủ thông tin
5. Trả lời theo ngôn ngữ câu hỏi của người dùng

## Giải thích code
Code sẽ:
1. Tạo hàm `build_naive_rag_prompt()`
2. Nhận đầu vào là `question` và `retrieved_df`
3. Gọi `format_context()` để tạo context có nguồn
4. Ghép thành prompt hoàn chỉnh
5. Lưu prompt template vào `artifacts/prompts/naive_rag_prompt_template.txt`
6. Test thử bằng câu hỏi về company handbook

## Output mong đợi
Bạn cần thấy prompt hoàn chỉnh gồm:
- phần hướng dẫn cho LLM
- câu hỏi người dùng
- retrieved context
- yêu cầu định dạng câu trả lời

In [4]:
def build_naive_rag_prompt(question, retrieved_df):
    context = format_context(retrieved_df)

    prompt = f"""
Bạn là trợ lý RAG nội bộ doanh nghiệp.

Nhiệm vụ:
- Trả lời câu hỏi chỉ dựa trên phần CONTEXT được cung cấp.
- Không tự bịa thông tin ngoài CONTEXT.
- Nếu CONTEXT không đủ thông tin, hãy nói: "Tôi chưa tìm thấy đủ thông tin trong tài liệu được cung cấp. Vui lòng liên hệ HR/pháp chế để được xử lý chính xác hơn."
- Câu trả lời phải dễ hiểu, ngắn gọn và có trích nguồn.
- Nếu câu hỏi bằng tiếng Việt, trả lời bằng tiếng Việt.
- Nếu câu hỏi bằng tiếng Anh, trả lời bằng tiếng Anh.

QUESTION:
{question}

CONTEXT:
{context}

YÊU CẦU OUTPUT:
1. Câu trả lời
2. Nguồn đã dùng
3. Evidence ngắn gọn
""".strip()

    return prompt

question = "What is the company policy for managing work devices?"

prompt = build_naive_rag_prompt(question, retrieved_df)

prompt_path = PROMPT_DIR / "naive_rag_prompt_template.txt"
prompt_path.write_text(prompt, encoding="utf-8")

print("Đã lưu prompt mẫu:", prompt_path)
print(prompt[:4000])

Đã lưu prompt mẫu: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\prompts\naive_rag_prompt_template.txt
Bạn là trợ lý RAG nội bộ doanh nghiệp.

Nhiệm vụ:
- Trả lời câu hỏi chỉ dựa trên phần CONTEXT được cung cấp.
- Không tự bịa thông tin ngoài CONTEXT.
- Nếu CONTEXT không đủ thông tin, hãy nói: "Tôi chưa tìm thấy đủ thông tin trong tài liệu được cung cấp. Vui lòng liên hệ HR/pháp chế để được xử lý chính xác hơn."
- Câu trả lời phải dễ hiểu, ngắn gọn và có trích nguồn.
- Nếu câu hỏi bằng tiếng Việt, trả lời bằng tiếng Việt.
- Nếu câu hỏi bằng tiếng Anh, trả lời bằng tiếng Anh.

QUESTION:
What is the company policy for managing work devices?

CONTEXT:
[Source 1]
source_type: company_handbook
title: Managing Work Devices
parent_id: company_0004
chunk_id: company_0004_001
score: 0.8258

With Linux, we run Omarchy, which already comes with the standard configuration needed (full-disk encryption, firewall, etc). Here we lean on 1password to

# Cell 05 - Đóng gói Naive RAG pipeline

## Mục tiêu cell này
Tạo hàm `naive_rag_pipeline()` để chạy trọn bước Retrieval và Prompt Augmentation.

## Vì sao cần làm bước này?
Ở các cell trước, ta đã có từng phần riêng lẻ:
- `dense_retrieve_chunks()` để truy xuất context
- `format_context()` để format nguồn
- `build_naive_rag_prompt()` để tạo prompt cho LLM

Cell này gom các phần đó thành một pipeline duy nhất.

Khi người dùng nhập câu hỏi, hệ thống sẽ:
1. Truy xuất top-k chunks liên quan bằng Dense Retrieval
2. Format các chunks thành sources/evidence
3. Tạo prompt RAG hoàn chỉnh
4. Trả về retrieved sources và prompt

Ở bước này, ta chưa gọi LLM thật.  
Mục tiêu là kiểm tra phần RAG input đã chuẩn trước khi kết nối model sinh câu trả lời.

## Giải thích code
Code sẽ:
1. Tạo hàm `naive_rag_pipeline()`
2. Nhận câu hỏi, top-k và source filter
3. Gọi Dense Retrieval để lấy context
4. Gọi prompt builder để tạo prompt
5. Lưu prompt mới nhất vào `artifacts/prompts/latest_naive_rag_prompt.txt`
6. Test thử với câu hỏi về chính sách thiết bị làm việc

## Output mong đợi
Bạn cần thấy:
- câu hỏi đầu vào
- bảng retrieved sources
- đường dẫn prompt đã lưu
- preview prompt RAG

In [5]:
def naive_rag_pipeline(question, top_k=5, source_filter=None):
    retrieved_df = dense_retrieve_chunks(
        query=question,
        top_k=top_k,
        source_filter=source_filter
    )

    prompt = build_naive_rag_prompt(question, retrieved_df)

    return {
        "question": question,
        "retrieved_sources": retrieved_df,
        "prompt": prompt
    }

question = "What is the company policy for managing work devices?"

rag_output = naive_rag_pipeline(
    question=question,
    top_k=5,
    source_filter=["company_handbook"]
)

latest_prompt_path = PROMPT_DIR / "latest_naive_rag_prompt.txt"
latest_prompt_path.write_text(rag_output["prompt"], encoding="utf-8")

print("Question:", rag_output["question"])
print("Đã lưu prompt:", latest_prompt_path)

display(
    rag_output["retrieved_sources"][
        ["rank", "score", "source_type", "title", "parent_id", "chunk_id"]
    ]
)

print("\nPrompt preview:")
print(rag_output["prompt"][:3000])

Question: What is the company policy for managing work devices?
Đã lưu prompt: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\prompts\latest_naive_rag_prompt.txt


,rank,score,source_type,title,parent_id,chunk_id
0,1,0.825828,company_handbook,Managing Work Devices,company_0004,company_0004_001
1,2,0.825389,company_handbook,Managing Work Devices,company_0004,company_0004_000
2,3,0.819602,company_handbook,Managing Work Devices,company_0004,company_0004_002
3,4,0.788421,company_handbook,Moonlighting,company_0005,company_0005_002
4,5,0.783055,company_handbook,Readme,company_0008,company_0008_000



Prompt preview:
Bạn là trợ lý RAG nội bộ doanh nghiệp.

Nhiệm vụ:
- Trả lời câu hỏi chỉ dựa trên phần CONTEXT được cung cấp.
- Không tự bịa thông tin ngoài CONTEXT.
- Nếu CONTEXT không đủ thông tin, hãy nói: "Tôi chưa tìm thấy đủ thông tin trong tài liệu được cung cấp. Vui lòng liên hệ HR/pháp chế để được xử lý chính xác hơn."
- Câu trả lời phải dễ hiểu, ngắn gọn và có trích nguồn.
- Nếu câu hỏi bằng tiếng Việt, trả lời bằng tiếng Việt.
- Nếu câu hỏi bằng tiếng Anh, trả lời bằng tiếng Anh.

QUESTION:
What is the company policy for managing work devices?

CONTEXT:
[Source 1]
source_type: company_handbook
title: Managing Work Devices
parent_id: company_0004
chunk_id: company_0004_001
score: 0.8258

With Linux, we run Omarchy, which already comes with the standard configuration needed (full-disk encryption, firewall, etc). Here we lean on 1password to provide the employee onboarding/offboarding of access to credentials and our Tailscale VPN to control access to internal systems. We don't

# Cell 06 - Tạo output Naive RAG dạng Answer + Sources + Evidence

## Mục tiêu cell này
Tạo câu trả lời mẫu dựa trên các context đã truy xuất, theo đúng format của hệ thống RAG doanh nghiệp.

## Vì sao cần làm bước này?
Một hệ thống RAG không chỉ trả lời văn bản, mà cần hiển thị rõ:
- câu trả lời
- nguồn đã dùng
- evidence chunks
- điểm similarity

Ở bước này, ta chưa gọi LLM thật.  
Ta tạo một bản output dạng extractive/grounded để kiểm tra format hiển thị và đảm bảo hệ thống không tự bịa ngoài context.

Sau này, phần `answer` này sẽ được thay bằng câu trả lời do LLM sinh từ prompt đã tạo ở Cell 05.

## Giải thích code
Code sẽ:
1. Nhận kết quả từ `naive_rag_pipeline()`
2. Lấy các source đã truy xuất
3. Tạo câu trả lời mẫu dựa trên source đầu tiên
4. Trả về dictionary gồm:
   - `answer`
   - `sources`
   - `evidence`
   - `prompt`
5. Hiển thị kết quả theo format dễ đọc

## Output mong đợi
Bạn cần thấy output gồm:
- Câu trả lời mẫu
- Danh sách nguồn
- Evidence ngắn gọn từ các chunk đã truy xuất

In [6]:
def build_grounded_preview_answer(rag_output, max_evidence_chars=500):
    retrieved_df = rag_output["retrieved_sources"]

    if retrieved_df.empty:
        return {
            "answer": "Tôi chưa tìm thấy đủ thông tin trong tài liệu được cung cấp. Vui lòng liên hệ HR/pháp chế để được xử lý chính xác hơn.",
            "sources": [],
            "evidence": []
        }

    top_source = retrieved_df.iloc[0]

    answer = (
        "Dựa trên tài liệu được truy xuất, chính sách liên quan nằm trong "
        f"'{top_source['title']}'. Nội dung chính cần được trả lời dựa trên các evidence bên dưới. "
        "Đây là bản preview có kiểm soát nguồn; ở bước sau LLM sẽ dùng prompt để sinh câu trả lời tự nhiên hơn."
    )

    sources = []
    evidence = []

    for _, row in retrieved_df.iterrows():
        sources.append({
            "rank": int(row["rank"]),
            "source_type": row["source_type"],
            "title": row["title"],
            "parent_id": row["parent_id"],
            "chunk_id": row["chunk_id"],
            "score": round(float(row["score"]), 4)
        })

        evidence.append({
            "rank": int(row["rank"]),
            "title": row["title"],
            "evidence": str(row["chunk_text"])[:max_evidence_chars]
        })

    return {
        "answer": answer,
        "sources": sources,
        "evidence": evidence,
        "prompt": rag_output["prompt"]
    }

preview_output = build_grounded_preview_answer(rag_output)

print("ANSWER:")
print(preview_output["answer"])

print("\nSOURCES:")
display(pd.DataFrame(preview_output["sources"]))

print("\nEVIDENCE:")
for item in preview_output["evidence"][:3]:
    print(f"\n[Source {item['rank']}] {item['title']}")
    print(item["evidence"])

ANSWER:
Dựa trên tài liệu được truy xuất, chính sách liên quan nằm trong 'Managing Work Devices'. Nội dung chính cần được trả lời dựa trên các evidence bên dưới. Đây là bản preview có kiểm soát nguồn; ở bước sau LLM sẽ dùng prompt để sinh câu trả lời tự nhiên hơn.

SOURCES:


,rank,source_type,title,parent_id,chunk_id,score
0,1,company_handbook,Managing Work Devices,company_0004,company_0004_001,0.8258
1,2,company_handbook,Managing Work Devices,company_0004,company_0004_000,0.8254
2,3,company_handbook,Managing Work Devices,company_0004,company_0004_002,0.8196
3,4,company_handbook,Moonlighting,company_0005,company_0005_002,0.7884
4,5,company_handbook,Readme,company_0008,company_0008_000,0.7831



EVIDENCE:

[Source 1] Managing Work Devices
With Linux, we run Omarchy, which already comes with the standard configuration needed (full-disk encryption, firewall, etc). Here we lean on 1password to provide the employee onboarding/offboarding of access to credentials and our Tailscale VPN to control access to internal systems. We don't use a managed process like Kandji.

## Access to code and secrets
Knowing our devices are safe and secure allows us to entrust our work computers with access to sensitive systems like Queenbee, and our inte

[Source 2] Managing Work Devices
# Managing work devices
Everyone receives a new Mac when they join 37signals. We centrally manage and secure these devices with Kandji which reduces our exposure to security incidents. Kandji applies a standard configuration to every device (e.g. enable disk encryption, firewall, password rules), it installs essential apps (e.g. EncryptMe), and it will ensure the apps have the latest security updates applied. Kandji 

# Cell 07 - Test Naive RAG với câu hỏi pháp luật tiếng Việt

## Mục tiêu cell này
Kiểm tra Naive RAG trên câu hỏi tiếng Việt thuộc legal corpus.

## Vì sao cần làm bước này?
Ở các cell trước, ta đã test câu hỏi tiếng Anh về handbook nội bộ công ty.  
Tuy nhiên, project của mình có hai nguồn tri thức:
- handbook nội bộ công ty bằng tiếng Anh
- văn bản pháp luật Việt Nam bằng tiếng Việt

Vì vậy cần kiểm tra pipeline có truy xuất được nguồn pháp luật tiếng Việt hay không.

## Giải thích code
Code sẽ:
1. Nhập một câu hỏi tiếng Việt về lương thử việc
2. Dùng `naive_rag_pipeline()` để truy xuất top-k legal chunks
3. Tạo prompt RAG hoàn chỉnh
4. Tạo output preview gồm Answer + Sources + Evidence
5. Hiển thị các nguồn pháp luật được truy xuất

## Output mong đợi
Bạn cần thấy các source có `source_type = legal` và nội dung liên quan đến thử việc, tiền lương thử việc hoặc quy định lao động.

In [7]:
question_vi = "Lương thử việc được quy định như thế nào?"

rag_output_vi = naive_rag_pipeline(
    question=question_vi,
    top_k=5,
    source_filter=["legal"]
)

preview_output_vi = build_grounded_preview_answer(rag_output_vi)

latest_prompt_vi_path = PROMPT_DIR / "latest_naive_rag_prompt_vi_legal.txt"
latest_prompt_vi_path.write_text(rag_output_vi["prompt"], encoding="utf-8")

print("QUESTION:")
print(question_vi)

print("\nĐã lưu prompt:", latest_prompt_vi_path)

print("\nSOURCES:")
display(
    rag_output_vi["retrieved_sources"][
        ["rank", "score", "source_type", "title", "parent_id", "chunk_id"]
    ]
)

print("\nANSWER PREVIEW:")
print(preview_output_vi["answer"])

print("\nEVIDENCE:")
for item in preview_output_vi["evidence"][:3]:
    print(f"\n[Source {item['rank']}] {item['title']}")
    print(item["evidence"])

QUESTION:
Lương thử việc được quy định như thế nào?

Đã lưu prompt: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\prompts\latest_naive_rag_prompt_vi_legal.txt

SOURCES:


,rank,score,source_type,title,parent_id,chunk_id
0,1,0.876999,legal,legal_cid_116782,116782,legal_116782_000
1,2,0.874766,legal,legal_cid_61953,61953,legal_61953_000
2,3,0.873931,legal,legal_cid_172654,172654,legal_172654_001
3,4,0.871711,legal,legal_cid_175106,175106,legal_175106_000
4,5,0.867452,legal,legal_cid_36069,36069,legal_36069_000



ANSWER PREVIEW:
Dựa trên tài liệu được truy xuất, chính sách liên quan nằm trong 'legal_cid_116782'. Nội dung chính cần được trả lời dựa trên các evidence bên dưới. Đây là bản preview có kiểm soát nguồn; ở bước sau LLM sẽ dùng prompt để sinh câu trả lời tự nhiên hơn.

EVIDENCE:

[Source 1] legal_cid_116782
Vi phạm quy định về thử việc
1. Phạt tiền từ 500.000 đồng đến 1.000.000 đồng đối với người sử dụng lao động có một trong các hành vi sau đây:
a) Yêu cầu thử việc đối với người lao động làm việc theo hợp đồng lao động có thời hạn dưới 01 tháng;
...

[Source 2] legal_cid_61953
“Điều 26. Tiền lương trong thời gian thử việc
 Tiền lương của người lao động trong thời gian thử việc do hai bên thoả thuận nhưng ít nhất phải bằng 85% mức lương của công việc đó.”

[Source 3] legal_cid_172654
bắt buộc, bảo hiểm y tế, bảo hiểm thất nghiệp. 
- Theo quy định của Luật Bảo hiểm xã hội thì người thử việc theo hợp đồng thử việc không thuộc đối tượng tham gia bảo hiểm xã hội bắt buộc"; khoản 3 Điều 168

# Cell 08 - Tạo Naive RAG truy xuất song song handbook nội bộ và pháp luật Việt Nam

## Mục tiêu cell này
Tạo pipeline Naive RAG có khả năng truy xuất từ hai nguồn riêng biệt:
- `company_handbook`: chính sách nội bộ công ty
- `legal`: quy định pháp luật Việt Nam

## Vì sao cần làm bước này?
Đề tài của mình không chỉ là chatbot hỏi luật chung, mà là trợ lý giúp đối chiếu chính sách nội bộ doanh nghiệp với quy định pháp luật Việt Nam.

Ví dụ người dùng hỏi:
"Công ty có chính sách về thiết bị làm việc, khi áp dụng ở Việt Nam có cần lưu ý gì không?"

Hệ thống cần lấy:
1. Evidence từ handbook nội bộ công ty
2. Evidence từ văn bản pháp luật Việt Nam nếu có liên quan

Việc truy xuất riêng từng nguồn giúp tránh tình trạng một nguồn quá lớn lấn át nguồn còn lại.  
Trong corpus hiện tại, legal có hơn 91,000 chunks, còn handbook chỉ có 95 chunks. Nếu search chung, legal có thể áp đảo kết quả.

## Giải thích code
Code sẽ:
1. Tạo hàm `dual_source_naive_rag_pipeline()`
2. Truy xuất top-k chunks từ `company_handbook`
3. Truy xuất top-k chunks từ `legal`
4. Gộp hai nhóm source lại
5. Tạo prompt RAG chung
6. Test thử bằng câu hỏi tiếng Việt có tính đối chiếu policy và pháp luật

## Output mong đợi
Bạn cần thấy bảng retrieved sources gồm cả:
- `company_handbook`
- `legal`

In [8]:
def dual_source_naive_rag_pipeline(question, company_top_k=3, legal_top_k=3):
    company_df = dense_retrieve_chunks(
        query=question,
        top_k=company_top_k,
        source_filter=["company_handbook"],
        search_k=500
    )

    legal_df = dense_retrieve_chunks(
        query=question,
        top_k=legal_top_k,
        source_filter=["legal"],
        search_k=500
    )

    retrieved_df = pd.concat([company_df, legal_df], ignore_index=True)
    retrieved_df["rank"] = range(1, len(retrieved_df) + 1)

    prompt = build_naive_rag_prompt(question, retrieved_df)

    return {
        "question": question,
        "retrieved_sources": retrieved_df,
        "prompt": prompt
    }

cross_question = "Nếu công ty áp dụng chính sách quản lý thiết bị làm việc cho nhân viên Việt Nam thì cần lưu ý gì?"

cross_rag_output = dual_source_naive_rag_pipeline(
    question=cross_question,
    company_top_k=3,
    legal_top_k=3
)

cross_prompt_path = PROMPT_DIR / "latest_naive_rag_prompt_cross_policy.txt"
cross_prompt_path.write_text(cross_rag_output["prompt"], encoding="utf-8")

print("QUESTION:")
print(cross_question)

print("\nĐã lưu prompt:", cross_prompt_path)

display(
    cross_rag_output["retrieved_sources"][
        ["rank", "score", "source_type", "title", "parent_id", "chunk_id"]
    ]
)

print("\nPROMPT PREVIEW:")
print(cross_rag_output["prompt"][:4000])

QUESTION:
Nếu công ty áp dụng chính sách quản lý thiết bị làm việc cho nhân viên Việt Nam thì cần lưu ý gì?

Đã lưu prompt: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\prompts\latest_naive_rag_prompt_cross_policy.txt


,rank,score,source_type,title,parent_id,chunk_id
0,1,0.852841,legal,legal_cid_56198,56198,legal_56198_001
1,2,0.851287,legal,legal_cid_117969,117969,legal_117969_000
2,3,0.850514,legal,legal_cid_152390,152390,legal_152390_000



PROMPT PREVIEW:
Bạn là trợ lý RAG nội bộ doanh nghiệp.

Nhiệm vụ:
- Trả lời câu hỏi chỉ dựa trên phần CONTEXT được cung cấp.
- Không tự bịa thông tin ngoài CONTEXT.
- Nếu CONTEXT không đủ thông tin, hãy nói: "Tôi chưa tìm thấy đủ thông tin trong tài liệu được cung cấp. Vui lòng liên hệ HR/pháp chế để được xử lý chính xác hơn."
- Câu trả lời phải dễ hiểu, ngắn gọn và có trích nguồn.
- Nếu câu hỏi bằng tiếng Việt, trả lời bằng tiếng Việt.
- Nếu câu hỏi bằng tiếng Anh, trả lời bằng tiếng Anh.

QUESTION:
Nếu công ty áp dụng chính sách quản lý thiết bị làm việc cho nhân viên Việt Nam thì cần lưu ý gì?

CONTEXT:
[Source 1]
source_type: legal
title: legal_cid_56198
parent_id: 56198
chunk_id: legal_56198_001
score: 0.8528

việc, người sử dụng lao động phải:
a) Thông báo công khai cho người lao động tại nơi quan trắc môi trường lao động và nơi được kiểm tra, đánh giá, quản lý yếu tố nguy hiểm;
b) Cung cấp thông tin khi tổ chức công đoàn, cơ quan, tổ chức có thẩm quyền yêu cầu;
c) Có biện pháp 

Kết quả này cho thấy pipeline dual-source chưa đạt mục tiêu, vì prompt preview chỉ thấy source_type: legal, chưa thấy company_handbook.

Nguyên nhân không phải FAISS lỗi, mà do câu hỏi tiếng Việt:

Nếu công ty áp dụng chính sách quản lý thiết bị làm việc cho nhân viên Việt Nam thì cần lưu ý gì?

khi search toàn corpus, top kết quả bị legal corpus áp đảo vì legal có hơn 91,353 chunks, còn company handbook chỉ có 95 chunks. Hàm hiện tại search trước rồi mới lọc source_type, nên có thể không lấy đủ company chunks.

# Cell 09 - Sửa hàm Dense Retrieval để truy xuất đủ từng nguồn

## Mục tiêu cell này
Cải thiện hàm `dense_retrieve_chunks()` để khi truy xuất theo từng nguồn dữ liệu, hệ thống vẫn lấy đủ context cần thiết.

## Vì sao cần làm bước này?
Corpus hiện tại bị lệch kích thước rất mạnh:
- `legal`: 91,353 chunks
- `company_handbook`: 95 chunks

Nếu search chung toàn corpus rồi mới lọc theo `source_type`, các kết quả từ `legal` có thể áp đảo, làm cho `company_handbook` không xuất hiện trong top kết quả.

Điều này không phù hợp với đề tài của mình, vì CrossPolicyRAG cần truy xuất song song:
1. chính sách nội bộ công ty
2. quy định pháp luật Việt Nam

## Giải thích code
Code sẽ định nghĩa lại `dense_retrieve_chunks()` theo hướng robust hơn:
1. Encode câu hỏi thành vector
2. Search FAISS với `search_k` ban đầu
3. Lọc theo `source_filter`
4. Nếu chưa đủ `top_k`, tự tăng `search_k`
5. Lặp lại cho đến khi đủ kết quả hoặc đạt giới hạn tối đa
6. Trả về bảng chunks đã truy xuất

Nhờ vậy, khi hỏi câu tiếng Việt nhưng cần lấy cả handbook tiếng Anh, hệ thống vẫn có khả năng tìm được company chunks.

## Output mong đợi
Khi test lại câu hỏi CrossPolicyRAG, bảng retrieved sources cần có cả:
- `company_handbook`
- `legal`

In [9]:
def dense_retrieve_chunks(query, top_k=5, source_filter=None, search_k=500, max_search_k=None):
    if max_search_k is None:
        max_search_k = len(chunks_df)

    query_embedding = embedding_model.encode(
        ["query: " + str(query)],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    ).astype("float32")

    current_search_k = min(search_k, len(chunks_df))
    final_rows = []

    while current_search_k <= max_search_k:
        scores, indices = faiss_index.search(query_embedding, current_search_k)

        rows = []
        rank = 1

        for score, idx in zip(scores[0], indices[0]):
            if idx == -1:
                continue

            row = chunks_df.iloc[idx]

            if source_filter is not None and row["source_type"] not in source_filter:
                continue

            rows.append({
                "rank": rank,
                "score": float(score),
                "corpus_id": row["corpus_id"],
                "chunk_id": row["chunk_id"],
                "parent_id": row["parent_id"],
                "source_type": row["source_type"],
                "title": row["title"],
                "language": row["language"],
                "chunk_text": row["chunk_text"]
            })

            rank += 1

            if len(rows) >= top_k:
                final_rows = rows
                break

        if len(final_rows) >= top_k:
            break

        current_search_k *= 2

        if current_search_k > max_search_k and rows:
            final_rows = rows
            break

    return pd.DataFrame(final_rows)


cross_question = "Nếu công ty áp dụng chính sách quản lý thiết bị làm việc cho nhân viên Việt Nam thì cần lưu ý gì?"

test_company_df = dense_retrieve_chunks(
    query=cross_question,
    top_k=3,
    source_filter=["company_handbook"],
    search_k=500,
    max_search_k=len(chunks_df)
)

test_legal_df = dense_retrieve_chunks(
    query=cross_question,
    top_k=3,
    source_filter=["legal"],
    search_k=500,
    max_search_k=len(chunks_df)
)

print("Company retrieved:", len(test_company_df))
display(test_company_df[["rank", "score", "source_type", "title", "chunk_id"]])

print("\nLegal retrieved:", len(test_legal_df))
display(test_legal_df[["rank", "score", "source_type", "title", "chunk_id"]])

Company retrieved: 3


,rank,score,source_type,title,chunk_id
0,1,0.791058,company_handbook,Managing Work Devices,company_0004_001
1,2,0.773632,company_handbook,Managing Work Devices,company_0004_002
2,3,0.772039,company_handbook,Managing Work Devices,company_0004_000



Legal retrieved: 3


,rank,score,source_type,title,chunk_id
0,1,0.852841,legal,legal_cid_56198,legal_56198_001
1,2,0.851287,legal,legal_cid_117969,legal_117969_000
2,3,0.850514,legal,legal_cid_152390,legal_152390_000


# Cell 10 - Chạy lại CrossPolicyRAG sau khi sửa retrieval từng nguồn

## Mục tiêu cell này
Tạo lại prompt CrossPolicyRAG có đủ hai nhóm nguồn:
- chính sách nội bộ công ty
- quy định pháp luật Việt Nam

## Vì sao cần làm bước này?
Ở Cell 08, prompt bị lệch vì chỉ thấy nguồn pháp luật.  
Sau Cell 09, hàm retrieval đã được sửa để truy xuất đủ từng nguồn riêng biệt.

Cell này kiểm tra lại toàn bộ pipeline đúng với mục tiêu đề tài:
`chính sách nội bộ doanh nghiệp + đối chiếu quy định pháp luật Việt Nam`.

## Giải thích code
Code sẽ:
1. Gọi lại `dual_source_naive_rag_pipeline()`
2. Truy xuất 3 chunks từ company handbook
3. Truy xuất 3 chunks từ legal corpus
4. Tạo prompt RAG hoàn chỉnh
5. Lưu prompt vào file
6. Hiển thị danh sách nguồn đã truy xuất

## Output mong đợi
Bảng retrieved sources phải có cả:
- `company_handbook`
- `legal`

In [10]:
cross_question = "Nếu công ty áp dụng chính sách quản lý thiết bị làm việc cho nhân viên Việt Nam thì cần lưu ý gì?"

cross_rag_output = dual_source_naive_rag_pipeline(
    question=cross_question,
    company_top_k=3,
    legal_top_k=3
)

cross_prompt_path = PROMPT_DIR / "latest_naive_rag_prompt_cross_policy_fixed.txt"
cross_prompt_path.write_text(cross_rag_output["prompt"], encoding="utf-8")

print("QUESTION:")
print(cross_question)

print("\nĐã lưu prompt:", cross_prompt_path)

print("\nRetrieved sources:")
display(
    cross_rag_output["retrieved_sources"][
        ["rank", "score", "source_type", "title", "parent_id", "chunk_id"]
    ]
)

print("\nSố lượng theo source_type:")
display(cross_rag_output["retrieved_sources"]["source_type"].value_counts().reset_index())

print("\nPROMPT PREVIEW:")
print(cross_rag_output["prompt"][:5000])

QUESTION:
Nếu công ty áp dụng chính sách quản lý thiết bị làm việc cho nhân viên Việt Nam thì cần lưu ý gì?

Đã lưu prompt: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\prompts\latest_naive_rag_prompt_cross_policy_fixed.txt

Retrieved sources:


,rank,score,source_type,title,parent_id,chunk_id
0,1,0.791058,company_handbook,Managing Work Devices,company_0004,company_0004_001
1,2,0.773632,company_handbook,Managing Work Devices,company_0004,company_0004_002
2,3,0.772039,company_handbook,Managing Work Devices,company_0004,company_0004_000
3,4,0.852841,legal,legal_cid_56198,56198,legal_56198_001
4,5,0.851287,legal,legal_cid_117969,117969,legal_117969_000
5,6,0.850514,legal,legal_cid_152390,152390,legal_152390_000



Số lượng theo source_type:


,source_type,count
0,company_handbook,3
1,legal,3



PROMPT PREVIEW:
Bạn là trợ lý RAG nội bộ doanh nghiệp.

Nhiệm vụ:
- Trả lời câu hỏi chỉ dựa trên phần CONTEXT được cung cấp.
- Không tự bịa thông tin ngoài CONTEXT.
- Nếu CONTEXT không đủ thông tin, hãy nói: "Tôi chưa tìm thấy đủ thông tin trong tài liệu được cung cấp. Vui lòng liên hệ HR/pháp chế để được xử lý chính xác hơn."
- Câu trả lời phải dễ hiểu, ngắn gọn và có trích nguồn.
- Nếu câu hỏi bằng tiếng Việt, trả lời bằng tiếng Việt.
- Nếu câu hỏi bằng tiếng Anh, trả lời bằng tiếng Anh.

QUESTION:
Nếu công ty áp dụng chính sách quản lý thiết bị làm việc cho nhân viên Việt Nam thì cần lưu ý gì?

CONTEXT:
[Source 1]
source_type: company_handbook
title: Managing Work Devices
parent_id: company_0004
chunk_id: company_0004_001
score: 0.7911

With Linux, we run Omarchy, which already comes with the standard configuration needed (full-disk encryption, firewall, etc). Here we lean on 1password to provide the employee onboarding/offboarding of access to credentials and our Tailscale VPN to 

# Cell 11 - Tạo câu trả lời preview cho CrossPolicyRAG

## Mục tiêu cell này
Tạo câu trả lời mẫu cho câu hỏi đối chiếu chính sách nội bộ công ty với quy định pháp luật Việt Nam.

## Vì sao cần làm bước này?
Ở Cell 10, hệ thống đã truy xuất được cả hai nguồn:
- handbook nội bộ công ty
- tài liệu pháp luật Việt Nam

Tuy nhiên, để demo RAG dễ hiểu hơn, ta cần hiển thị kết quả cuối theo format:
1. Câu trả lời
2. Nguồn nội bộ đã dùng
3. Nguồn pháp luật đã dùng
4. Evidence ngắn gọn
5. Cảnh báo nếu cần HR/pháp chế kiểm tra thêm

Ở bước này, ta vẫn chưa gọi LLM thật.  
Câu trả lời preview được tạo theo template có kiểm soát để chứng minh pipeline RAG đã có đủ nguồn và evidence.

## Giải thích code
Code sẽ:
1. Tách retrieved sources thành hai nhóm: `company_handbook` và `legal`
2. Tạo câu trả lời preview bằng tiếng Việt
3. Trích evidence ngắn từ các nguồn đã truy xuất
4. Lưu output vào file JSON để dùng lại cho demo hoặc báo cáo

## Output mong đợi
Bạn cần thấy câu trả lời có đủ:
- phần nhận định từ handbook nội bộ
- phần lưu ý khi áp dụng tại Việt Nam
- danh sách nguồn nội bộ
- danh sách nguồn pháp luật

In [11]:
import json

def build_cross_policy_preview(rag_output, max_evidence_chars=450):
    retrieved_df = rag_output["retrieved_sources"].copy()

    company_df = retrieved_df[retrieved_df["source_type"] == "company_handbook"].copy()
    legal_df = retrieved_df[retrieved_df["source_type"] == "legal"].copy()

    if company_df.empty and legal_df.empty:
        answer = "Tôi chưa tìm thấy đủ thông tin trong tài liệu được cung cấp. Vui lòng liên hệ HR/pháp chế để được xử lý chính xác hơn."
    elif company_df.empty:
        answer = "Tôi tìm thấy một số tài liệu pháp luật liên quan, nhưng chưa tìm thấy chính sách nội bộ công ty tương ứng. Vì vậy chưa đủ căn cứ để đối chiếu chính sách nội bộ với quy định Việt Nam."
    elif legal_df.empty:
        answer = "Tôi tìm thấy chính sách nội bộ công ty liên quan, nhưng chưa tìm thấy tài liệu pháp luật Việt Nam đủ rõ để đối chiếu. Vui lòng chuyển HR/pháp chế kiểm tra thêm trước khi áp dụng."
    else:
        answer = (
            "Theo handbook nội bộ, công ty có chính sách quản lý thiết bị làm việc, bao gồm việc cấu hình bảo mật, "
            "quản lý thiết bị được cấp, kiểm soát truy cập hệ thống nội bộ, và hạn chế lưu mã nguồn hoặc secrets trên thiết bị cá nhân. "
            "Khi áp dụng cho nhân viên Việt Nam, công ty nên đối chiếu thêm với quy định nội bộ tại Việt Nam và các tài liệu pháp luật liên quan đến "
            "quản lý trang thiết bị, an toàn nơi làm việc, bàn giao/cập nhật thiết bị và trách nhiệm sử dụng tài sản. "
            "Vì các nguồn pháp luật truy xuất được chưa đủ để kết luận toàn diện, HR/pháp chế nên kiểm tra thêm hợp đồng lao động, biên bản bàn giao thiết bị và quy chế tài sản trước khi áp dụng chính sách."
        )

    sources = []
    evidence = []

    for _, row in retrieved_df.iterrows():
        sources.append({
            "rank": int(row["rank"]),
            "source_type": row["source_type"],
            "title": row["title"],
            "parent_id": row["parent_id"],
            "chunk_id": row["chunk_id"],
            "score": round(float(row["score"]), 4)
        })

        evidence.append({
            "rank": int(row["rank"]),
            "source_type": row["source_type"],
            "title": row["title"],
            "evidence": str(row["chunk_text"])[:max_evidence_chars]
        })

    return {
        "question": rag_output["question"],
        "answer": answer,
        "sources": sources,
        "evidence": evidence
    }

cross_preview = build_cross_policy_preview(cross_rag_output)

cross_preview_path = PROMPT_DIR / "cross_policy_preview_output.json"

with open(cross_preview_path, "w", encoding="utf-8") as f:
    json.dump(cross_preview, f, ensure_ascii=False, indent=2)

print("QUESTION:")
print(cross_preview["question"])

print("\nANSWER:")
print(cross_preview["answer"])

print("\nSOURCES:")
display(pd.DataFrame(cross_preview["sources"]))

print("\nEVIDENCE:")
for item in cross_preview["evidence"]:
    print(f"\n[Source {item['rank']}] {item['source_type']} | {item['title']}")
    print(item["evidence"])

print("\nĐã lưu:", cross_preview_path)

QUESTION:
Nếu công ty áp dụng chính sách quản lý thiết bị làm việc cho nhân viên Việt Nam thì cần lưu ý gì?

ANSWER:
Theo handbook nội bộ, công ty có chính sách quản lý thiết bị làm việc, bao gồm việc cấu hình bảo mật, quản lý thiết bị được cấp, kiểm soát truy cập hệ thống nội bộ, và hạn chế lưu mã nguồn hoặc secrets trên thiết bị cá nhân. Khi áp dụng cho nhân viên Việt Nam, công ty nên đối chiếu thêm với quy định nội bộ tại Việt Nam và các tài liệu pháp luật liên quan đến quản lý trang thiết bị, an toàn nơi làm việc, bàn giao/cập nhật thiết bị và trách nhiệm sử dụng tài sản. Vì các nguồn pháp luật truy xuất được chưa đủ để kết luận toàn diện, HR/pháp chế nên kiểm tra thêm hợp đồng lao động, biên bản bàn giao thiết bị và quy chế tài sản trước khi áp dụng chính sách.

SOURCES:


,rank,source_type,title,parent_id,chunk_id,score
0,1,company_handbook,Managing Work Devices,company_0004,company_0004_001,0.7911
1,2,company_handbook,Managing Work Devices,company_0004,company_0004_002,0.7736
2,3,company_handbook,Managing Work Devices,company_0004,company_0004_000,0.7720
3,4,legal,legal_cid_56198,56198,legal_56198_001,0.8528
4,5,legal,legal_cid_117969,117969,legal_117969_000,0.8513
5,6,legal,legal_cid_152390,152390,legal_152390_000,0.8505



EVIDENCE:

[Source 1] company_handbook | Managing Work Devices
With Linux, we run Omarchy, which already comes with the standard configuration needed (full-disk encryption, firewall, etc). Here we lean on 1password to provide the employee onboarding/offboarding of access to credentials and our Tailscale VPN to control access to internal systems. We don't use a managed process like Kandji.

## Access to code and secrets
Knowing our devices are safe and secure allows us to entrust our work computers with acces

[Source 2] company_handbook | Managing Work Devices
## Mobile devices and Windows
Devices running Android, iOS/iPadOS, and Windows are currently unmanaged. It’s fine to install our BC4 and HEY apps on these devices to access work projects and email, but since they’re unmanaged – and therefore ‘untrusted’ – it’s not okay to store 37signals code or secrets on them. If you're coding or accessing secure systems, you should be doing so on a Kandji-managed Mac or an Omarchy Linux machi

# Cell 12 - Thêm confidence gate để tránh trả lời khi context yếu

## Mục tiêu cell này
Tạo cơ chế kiểm tra độ tin cậy của context trước khi hệ thống trả lời.

## Vì sao cần làm bước này?
Naive RAG luôn trả về top-k tài liệu gần nhất, kể cả khi câu hỏi không liên quan đến tài liệu nội bộ hoặc pháp luật.  
Nếu không kiểm tra độ tin cậy, hệ thống có thể lấy nhầm tài liệu và tạo câu trả lời sai.

Vì vậy, cell này thêm một bước kiểm tra:
- Nếu context có điểm similarity đủ cao → cho phép trả lời
- Nếu context yếu → từ chối trả lời và khuyên liên hệ HR/pháp chế

## Giải thích code
Code sẽ:
1. Tạo hàm `check_context_confidence()`
2. Tạo hàm `safe_naive_rag_preview()`
3. Chạy thử với một câu liên quan đến chính sách thiết bị
4. Chạy thử với một câu không liên quan đến hệ thống
5. Kiểm tra hệ thống có biết từ chối khi context yếu hay không

## Output mong đợi
Câu liên quan sẽ có `is_confident = True`.  
Câu không liên quan có thể có `is_confident = False` nếu điểm truy xuất thấp hơn ngưỡng.

In [13]:
def check_context_confidence(retrieved_df, min_score=0.78):
    if retrieved_df.empty:
        return {
            "is_confident": False,
            "reason": "Không truy xuất được tài liệu liên quan.",
            "max_score": 0.0
        }

    max_score = float(retrieved_df["score"].max())

    if max_score < min_score:
        return {
            "is_confident": False,
            "reason": f"Điểm truy xuất cao nhất ({max_score:.4f}) thấp hơn ngưỡng {min_score}.",
            "max_score": max_score
        }

    return {
        "is_confident": True,
        "reason": f"Context đạt ngưỡng tin cậy với max_score = {max_score:.4f}.",
        "max_score": max_score
    }


def safe_naive_rag_preview(question, source_filter=None, top_k=5, min_score=0.78):
    rag_output = naive_rag_pipeline(
        question=question,
        top_k=top_k,
        source_filter=source_filter
    )

    confidence = check_context_confidence(
        rag_output["retrieved_sources"],
        min_score=min_score
    )

    if not confidence["is_confident"]:
        return {
            "question": question,
            "answer": "Tôi chưa tìm thấy đủ thông tin trong tài liệu được cung cấp. Vui lòng liên hệ HR/pháp chế để được xử lý chính xác hơn.",
            "confidence": confidence,
            "sources": [],
            "evidence": []
        }

    answer = build_grounded_preview_answer(rag_output)
    answer["question"] = question
    answer["confidence"] = confidence

    return answer


related_question = "What is the company policy for managing work devices?"
unrelated_question = "Cách nấu phở bò ngon tại nhà như thế nào?"

related_output = safe_naive_rag_preview(
    question=related_question,
    source_filter=["company_handbook"],
    top_k=5,
    min_score=0.78
)

unrelated_output = safe_naive_rag_preview(
    question=unrelated_question,
    source_filter=["company_handbook"],
    top_k=5,
    min_score=0.78
)

print("RELATED QUESTION:")
print(related_output["question"])
print("Confidence:", related_output["confidence"])
print("Answer:", related_output["answer"])

print("\n" + "=" * 80 + "\n")

print("UNRELATED QUESTION:")
print(unrelated_output["question"])
print("Confidence:", unrelated_output["confidence"])
print("Answer:", unrelated_output["answer"])

RELATED QUESTION:
What is the company policy for managing work devices?
Confidence: {'is_confident': True, 'reason': 'Context đạt ngưỡng tin cậy với max_score = 0.8258.', 'max_score': 0.8258280754089355}
Answer: Dựa trên tài liệu được truy xuất, chính sách liên quan nằm trong 'Managing Work Devices'. Nội dung chính cần được trả lời dựa trên các evidence bên dưới. Đây là bản preview có kiểm soát nguồn; ở bước sau LLM sẽ dùng prompt để sinh câu trả lời tự nhiên hơn.


UNRELATED QUESTION:
Cách nấu phở bò ngon tại nhà như thế nào?
Confidence: {'is_confident': False, 'reason': 'Điểm truy xuất cao nhất (0.7533) thấp hơn ngưỡng 0.78.', 'max_score': 0.7532504200935364}
Answer: Tôi chưa tìm thấy đủ thông tin trong tài liệu được cung cấp. Vui lòng liên hệ HR/pháp chế để được xử lý chính xác hơn.


# Cell 13 - Lưu kết quả demo Naive RAG và confidence gate

## Mục tiêu cell này
Lưu lại các kết quả demo quan trọng của notebook `06_naive_rag.ipynb`.

## Vì sao cần làm bước này?
Notebook 06 đã tạo phiên bản Naive RAG đầu tiên, gồm:
- Dense Retrieval để lấy context
- Prompt RAG có nguồn
- Preview answer dạng Answer + Sources + Evidence
- CrossPolicyRAG truy xuất cả handbook nội bộ và pháp luật Việt Nam
- Confidence gate để từ chối khi context yếu

Các kết quả này cần được lưu lại để:
- dùng trong báo cáo
- dùng trong demo Streamlit sau này
- so sánh với Hybrid RAG và Reranker RAG ở các notebook sau

## Giải thích code
Code sẽ:
1. Gom các output demo chính thành một dictionary
2. Lưu kết quả vào `artifacts/prompts/naive_rag_demo_summary.json`
3. Hiển thị tóm tắt các case đã test

## Output mong đợi
Bạn cần thấy file JSON được lưu thành công và bảng tóm tắt các demo case.

In [14]:
naive_rag_demo_summary = {
    "notebook": "06_naive_rag.ipynb",
    "retrieval_method": "Dense Retrieval with multilingual-e5-base",
    "main_outputs": {
        "company_prompt": str(PROMPT_DIR / "latest_naive_rag_prompt.txt"),
        "legal_prompt": str(PROMPT_DIR / "latest_naive_rag_prompt_vi_legal.txt"),
        "cross_policy_prompt": str(PROMPT_DIR / "latest_naive_rag_prompt_cross_policy_fixed.txt"),
        "cross_policy_preview": str(PROMPT_DIR / "cross_policy_preview_output.json")
    },
    "demo_cases": [
        {
            "case": "company_handbook_question",
            "question": related_question,
            "source_filter": "company_handbook",
            "is_confident": related_output["confidence"]["is_confident"],
            "max_score": related_output["confidence"]["max_score"]
        },
        {
            "case": "unrelated_question",
            "question": unrelated_question,
            "source_filter": "company_handbook",
            "is_confident": unrelated_output["confidence"]["is_confident"],
            "max_score": unrelated_output["confidence"]["max_score"]
        },
        {
            "case": "cross_policy_question",
            "question": cross_question,
            "source_filter": "company_handbook + legal",
            "num_sources": len(cross_rag_output["retrieved_sources"]),
            "source_distribution": cross_rag_output["retrieved_sources"]["source_type"].value_counts().to_dict()
        }
    ]
}

summary_path = PROMPT_DIR / "naive_rag_demo_summary.json"

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(naive_rag_demo_summary, f, ensure_ascii=False, indent=2)

demo_cases_df = pd.DataFrame(naive_rag_demo_summary["demo_cases"])

print("Đã lưu:", summary_path)
display(demo_cases_df)

Đã lưu: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\prompts\naive_rag_demo_summary.json


,case,question,source_filter,is_confident,max_score,num_sources,source_distribution
0,company_handbook_question,What is the company policy for managing work d...,company_handbook,True,0.825828,NaN,NaN
1,unrelated_question,Cách nấu phở bò ngon tại nhà như thế nào?,company_handbook,False,0.753250,NaN,NaN
2,cross_policy_question,Nếu công ty áp dụng chính sách quản lý thiết b...,company_handbook + legal,NaN,NaN,6.0,"{'company_handbook': 3, 'legal': 3}"


# Cell 14 - Kiểm tra cuối các file Naive RAG đã tạo

## Mục tiêu cell này
Kiểm tra toàn bộ file đầu ra quan trọng của notebook `06_naive_rag.ipynb`.

## Vì sao cần làm bước này?
Notebook 06 đã xây dựng phiên bản Naive RAG đầu tiên.  
Các kết quả của notebook này sẽ được dùng để:
- demo pipeline RAG cơ bản
- so sánh với Hybrid RAG ở notebook 07
- so sánh với Reranker RAG ở notebook 08
- dùng lại prompt và preview output cho Streamlit demo sau này

## Giải thích code
Code sẽ kiểm tra các file:
- prompt handbook nội bộ
- prompt legal tiếng Việt
- prompt CrossPolicyRAG
- output preview CrossPolicyRAG
- summary demo Naive RAG

Sau đó hiển thị trạng thái `OK` hoặc `MISSING`.

## Output mong đợi
Tất cả file cần có trạng thái `OK`.

In [15]:
required_naive_rag_files = [
    PROMPT_DIR / "latest_naive_rag_prompt.txt",
    PROMPT_DIR / "latest_naive_rag_prompt_vi_legal.txt",
    PROMPT_DIR / "latest_naive_rag_prompt_cross_policy_fixed.txt",
    PROMPT_DIR / "cross_policy_preview_output.json",
    PROMPT_DIR / "naive_rag_demo_summary.json"
]

naive_rag_check_df = pd.DataFrame([
    {
        "file": str(path.relative_to(PROJECT)),
        "status": "OK" if path.exists() else "MISSING",
        "size_kb": round(path.stat().st_size / 1024, 2) if path.exists() else 0
    }
    for path in required_naive_rag_files
])

display(naive_rag_check_df)

print("Tổng file OK:", (naive_rag_check_df["status"] == "OK").sum(), "/", len(naive_rag_check_df))

,file,status,size_kb
0,artifacts\prompts\latest_naive_rag_prompt.txt,OK,6.38
1,artifacts\prompts\latest_naive_rag_prompt_vi_l...,OK,5.74
2,artifacts\prompts\latest_naive_rag_prompt_cros...,OK,7.27
3,artifacts\prompts\cross_policy_preview_output....,OK,6.10
4,artifacts\prompts\naive_rag_demo_summary.json,OK,1.70


Tổng file OK: 5 / 5




## 1. File 06 làm gì?

File 06 tạo phiên bản RAG đầu tiên, gọi là **Naive RAG**.

Nói đơn giản:

```text
Người dùng hỏi
→ Dense Retrieval tìm tài liệu liên quan
→ ghép tài liệu vào prompt
→ chuẩn bị cho LLM trả lời có nguồn
```

File 06 **chưa phải bản RAG mạnh nhất**. Nó là bản nền để sau này so sánh với:

```text
07_hybrid_rag.ipynb
08_reranker_rag.ipynb
09_corrective_rag.ipynb
```

---

## 2. Naive RAG là gì?

Naive RAG là RAG đơn giản nhất:

```text
Question
→ Retrieve top-k chunks
→ Build prompt
→ Generate answer
```

Trong file 06, mình mới làm tới phần:

```text
Question
→ Retrieve context
→ Build prompt
→ Preview Answer + Sources + Evidence
```

Chưa gọi LLM thật. Vì trước khi gọi LLM, mình cần chắc chắn phần **retrieval + prompt + source format** hoạt động đúng.

---

## 3. Ví dụ thực tế

Người dùng hỏi:

```text
What is the company policy for managing work devices?
```

Hệ thống truy xuất được các nguồn:

```text
Managing Work Devices - chunk 1
Managing Work Devices - chunk 0
Managing Work Devices - chunk 2
```

Các nguồn này nói về:

```text
thiết bị công ty cấp
quản lý bảo mật thiết bị
không lưu code/secrets trên thiết bị cá nhân
thiết bị có thể bị remote wipe khi mất hoặc khi nhân viên rời công ty
```

Sau đó hệ thống tạo prompt kiểu:

```text
Bạn là trợ lý RAG nội bộ doanh nghiệp.
Chỉ trả lời dựa trên CONTEXT.
Không bịa ngoài tài liệu.
Nếu không đủ thông tin thì nói chưa đủ thông tin.
Câu trả lời phải có nguồn.
```

Đây chính là phần giúp RAG khác ChatGPT thường: **nó bị ràng buộc bởi tài liệu truy xuất được**.

---

## 4. File 06 đã làm những bước nào?

### Bước 1: Load FAISS Dense Index

File 06 load lại các file từ notebook 05:

```text
indexes/faiss/faiss_index.index
indexes/faiss/dense_metadata.csv
```

Kết quả của bạn:

```text
Chunks metadata: 91,448
FAISS vectors: 91,448
Device: cuda
```

Tức là Dense Retrieval đã sẵn sàng.

---

### Bước 2: Tạo hàm dense retrieval

Mình tạo hàm:

```python
dense_retrieve_chunks()
```

Hàm này nhận câu hỏi rồi trả về top-k chunks liên quan.

Ví dụ với câu:

```text
What is the company policy for managing work devices?
```

nó trả về đúng nhóm:

```text
Managing Work Devices
```

---

### Bước 3: Format context

Mình tạo hàm:

```python
format_context()
```

Hàm này biến các chunks truy xuất được thành dạng rõ ràng:

```text
[Source 1]
source_type: company_handbook
title: Managing Work Devices
parent_id: company_0004
chunk_id: company_0004_001
score: 0.8258

Nội dung evidence...
```

Mục đích là để LLM biết rõ:

```text
nguồn nào
chunk nào
score bao nhiêu
evidence là gì
```

---

### Bước 4: Tạo prompt RAG

Mình tạo hàm:

```python
build_naive_rag_prompt()
```

Prompt yêu cầu LLM:

```text
Chỉ trả lời dựa trên context.
Không tự bịa.
Nếu thiếu thông tin thì từ chối.
Trả lời có nguồn.
```

Đây là phần rất quan trọng để giảm hallucination.

---

### Bước 5: Đóng gói pipeline

Mình tạo:

```python
naive_rag_pipeline()
```

Pipeline này làm trọn:

```text
question
→ dense retrieval
→ format context
→ build prompt
```

Nói dễ hiểu: chỉ cần nhập câu hỏi, hệ thống tự tạo prompt RAG hoàn chỉnh.

---

## 5. File 06 test được những kiểu câu hỏi nào?

### Case 1: Hỏi handbook nội bộ

Câu hỏi:

```text
What is the company policy for managing work devices?
```

Kết quả:

```text
Tìm được Managing Work Devices
Tạo prompt có source/evidence
```

---

### Case 2: Hỏi pháp luật tiếng Việt

Câu hỏi:

```text
Lương thử việc được quy định như thế nào?
```

Hệ thống tìm được source quan trọng:

```text
Điều 26. Tiền lương trong thời gian thử việc
Tiền lương thử việc ít nhất bằng 85% mức lương của công việc đó.
```

Đây là ví dụ rất tốt để chứng minh RAG tìm được evidence pháp luật.

---

### Case 3: Hỏi kiểu CrossPolicyRAG

Câu hỏi:

```text
Nếu công ty áp dụng chính sách quản lý thiết bị làm việc cho nhân viên Việt Nam thì cần lưu ý gì?
```

Ban đầu hệ thống bị lệch, chỉ lấy legal sources. Sau đó mình sửa hàm retrieval để lấy riêng từng nguồn.

Kết quả sau sửa:

```text
company_handbook: 3 sources
legal: 3 sources
```

Đây mới đúng mục tiêu đề tài:

```text
chính sách nội bộ công ty
+
quy định pháp luật Việt Nam
```

---

## 6. Vì sao phải sửa dual-source retrieval?

Vì corpus bị lệch rất mạnh:

```text
legal: 91,353 chunks
company_handbook: 95 chunks
```

Nếu search chung, legal dễ áp đảo handbook.

Nên mình sửa thành:

```text
tìm riêng 3 chunks từ company_handbook
tìm riêng 3 chunks từ legal
rồi gộp lại
```

Đây là quyết định đúng cho đề tài CrossPolicyRAG.

---

## 7. Confidence gate là gì?

Mình thêm bước:

```python
check_context_confidence()
```

Mục đích: không cho hệ thống trả lời khi tài liệu truy xuất yếu.

Ví dụ câu liên quan:

```text
What is the company policy for managing work devices?
```

Kết quả:

```text
max_score = 0.8258 > 0.78
→ đủ tin cậy
→ cho phép trả lời
```

Câu không liên quan:

```text
Cách nấu phở bò ngon tại nhà như thế nào?
```

Kết quả:

```text
max_score = 0.7533 < 0.78
→ context yếu
→ từ chối trả lời
```

Đây là điểm rất quan trọng trong RAG doanh nghiệp, vì hệ thống không nên cố trả lời khi không có tài liệu phù hợp.

---

## 8. File 06 đã tạo những file gì?

Bạn đã kiểm tra cuối và thấy:

```text
latest_naive_rag_prompt.txt                         OK
latest_naive_rag_prompt_vi_legal.txt                OK
latest_naive_rag_prompt_cross_policy_fixed.txt      OK
cross_policy_preview_output.json                    OK
naive_rag_demo_summary.json                         OK
```

Tổng:

```text
5 / 5 file OK
```

Nghĩa là file 06 đã hoàn tất.

---

## 9. Hạn chế của file 06

File 06 vẫn là **Naive RAG**, nên còn hạn chế:

```text
Chỉ dùng Dense Retrieval
Chưa kết hợp BM25
Chưa có reranker
Chưa gọi LLM thật để sinh answer tự nhiên
Legal sources đôi khi chưa phải căn cứ tốt nhất
Confidence gate còn đơn giản, chỉ dựa vào similarity score
```

Vì vậy cần file 07.

---

## 10. File 07 sẽ làm gì?

File 07 sẽ làm:

```text
Hybrid RAG = BM25 + Dense Retrieval
```

Lý do:

```text
Dense tốt hơn BM25 tổng thể,
nhưng BM25 vẫn cứu được 428 câu mà Dense bỏ sót.
```

Vậy file 07 sẽ kết hợp cả hai để tìm tài liệu tốt hơn.

---

## Chốt ngắn gọn

File 06 tạo bản RAG đầu tiên:

```text
User question
→ Dense Retrieval
→ Retrieved Sources
→ Evidence
→ Prompt RAG
→ Answer Preview
→ Confidence Gate
```

Nó chứng minh hệ thống đã có khả năng:

```text
hỏi handbook nội bộ
hỏi pháp luật Việt Nam
hỏi dạng đối chiếu policy + legal
từ chối khi câu hỏi không liên quan
```

Sau file 06, mình mới có nền để nâng cấp sang **Hybrid RAG** ở file 07.
